In [1]:
import torch
from torch import nn
from torch.nn import functional as F

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

import numpy as np

import tqdm as tqdm

import tiktoken

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


# Reading the input text file

In [3]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# Define tokens  and their encode/decode

In [4]:
# TokenizerName = 'Char'
TokenizerName = 'Tiktoken'

if TokenizerName=='Char':

    # set(text): find unique characteres
    # list(set(text)): convert to list
    # sorted(list(set(text))): sort
    Tokens = sorted(list(set(text)))
    NToken = len(Tokens)

    print('Number of tokens:', NToken)

    # encode: encode(list of character) = list of integer, index from Tokens list
    # decode: decode(list of integer) = list of chracter

    dict_c_to_i = {c:i_c for i_c, c in enumerate(Tokens)}
    dict_i_to_c = {i_c:c for i_c, c in enumerate(Tokens)}

    encode = lambda list_c: [dict_c_to_i[c] for c in list_c]
    decode = lambda list_i: ''.join([dict_i_to_c[i] for i in list_i])
    

if TokenizerName=='Tiktoken':
    
    enc = tiktoken.get_encoding("gpt2")

    NToken = enc.n_vocab

    encode = lambda list_c: enc.encode(list_c)
    decode = lambda list_i: enc.decode(list_i)


In [5]:
print('Number of Tokens:',NToken)
print(encode('My name is Jaesung'))
print(decode(encode('My name is Jaesung')))

Number of Tokens: 50257
[3666, 1438, 318, 13790, 274, 2150]
My name is Jaesung


# Now convert the text input into data

In [6]:
data = torch.tensor(encode(text), dtype=torch.long)

# First 90% of data for the training, Last 10% for the validation

In [7]:
tmp_train_index = int(len(data)*0.9)
print(len(data))

data_train = data[:tmp_train_index]
data_val = data[tmp_train_index:]

print(len(data_train), len(data_train)/len(data))
print(len(data_val), len(data_val)/len(data))

338025
304222 0.899998520819466
33803 0.10000147918053398


# Hyperparameters

In [25]:
# Size of the context
NContext = 256
# Size of the batch for each training
NBatch = 4

# Max iteration of the training
NMaxIteration = 100

# Evaluate the training every
EvalEvery = 1000
# Number of iteration for every evaluation step
EvalIteration = 200

# Learning rate
LearningRate = 3E-4

# Head
HeadSize = 64
NHead = 6
NEmbedding = HeadSize*NHead
# Layer
NLayer = 6

DropoutRate = 0.20

# Model

In [9]:
class Head(nn.Module):

    """ one head of self-attention """

    def __init__(self):

        super().__init__()

        self.key = nn.Linear(NEmbedding, HeadSize, bias=False)
        self.query = nn.Linear(NEmbedding, HeadSize, bias=False)
        self.value = nn.Linear(NEmbedding, HeadSize, bias=False)

        # https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer
        self.register_buffer('tril', torch.tril(torch.ones(NContext, NContext)))

        # During training, randomly zeroes some of the elements of the input tensor with probability
        # https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html#dropout
        self.dropout = nn.Dropout(DropoutRate)

    def forward(self, x):

        """
        x: (Batch, Context, Embedding)
        """

        # Dimensions
        D_Batch, D_Context, D_Embedding = x.shape

        # Query
        q = self.query(x) # (D_Batch, D_Context, HeadSize)

        # Key
        k = self.key(x)   # (D_Batch, D_Context, HeadSize)

        # Here we want to evaluate the attention scores ("affinities") 
        # I'll omit D_Batch.
        # For a given i_Context position, the query is q[i_Context,:] is a HeadSize-vector.
        # We also have key vector for each of j_Context position, k[j_Context,:].
        # We want a dot-product between query and key:
        #   Score[i_Context, j_Context] = q[i_Context,:] * k[j_Context,:] (dot product)
        #                               = q[i_Context,m]k[j_Context,m] (Einstein-summation)
        #                               = q[i_Context,m] k.T[m,j_Context] (Einstein-summation)
        #                               = (q @ k.T)[i_Context, j_Context]
        # So, the score is QK^T from the "Attention is all you need" paper

        # compute attention scores ("affinities")
        # q: (D_Batch, D_Context, HeadSize)
        # k: (D_Batch, D_Context, HeadSize)
        # -> k.transpose(-2, -1) = k.transpose(1, 2): (D_Batch, HeadSize, D_Context)
        # q @ k.transpose(-2, -1): (D_Batch, D_Context, D_Context)
        QKT = q @ k.transpose(-2,-1) * (D_Embedding**-0.5)
        
        # Now we want to mask
        # For each i_Context of query, we only fill j_Context from 0 to i_Context
        #          K
        #       1  0  0
        #  Q    1  1  0
        #       1  1  1
        # So absically, we want a lower-left triangle, and that is tril (tri-lower for tril, tri-upper for triu)
        # Our tril we regiestered is (NContext, NContext), but for a generating step, our input may be shorter than this
        # So we take first D_Context components of the tril
        # Then we fill -inf where the tril value is zero, so the upper-right off-dialgonals
        QKT = QKT.masked_fill(self.tril[:D_Context, :D_Context] == 0, float('-inf')) # (B, T, T)

        # Apply softmax on Key dimension (the last one)
        QKT = F.softmax(QKT, dim=-1)

        # Dropout
        QKT = self.dropout(QKT)

        # Value
        v = self.value(x) # (D_Batch, D_Context, HeadSize)

        # For each Context from QKT, it is a context-sized array of number;
        # QKT[i_Context,:] = [0.90, 0.05, 0.01, 0, ...., 0]
        # Then the Value vector is also cont-text sized array of Embedded vector;
        # V = [ [vec_0], [vec_1], [vec_2], ..., [vec_NContext]]
        # We want a weighted sum
        # QKT[i_Context,m] V[m]

        # QKT: (D_Batch, D_Context, D_Context)
        # v: (D_Batch, D_Context, HeadSize)
        # out: (D_Batch, D_Context, HeadSize)
        out = QKT @ v

        return out

In [10]:
class MultiHeadAttention(nn.Module):

    def __init__(self):

        super().__init__()
        # Multi head as parallel
        # Each head outputs (D_Batch, D_Context, HeadSize)
        # We will then concatenate NHead in the last dimension,
        # so becomes (D_Batch, D_Context, HeadSize*NHead) = (D_Batch, D_Context, NEmbedding)
        self.heads = nn.ModuleList([Head() for _ in range(NHead)])
        # Projection
        self.proj = nn.Linear(NEmbedding, NEmbedding)
        # Dropout
        self.dropout = nn.Dropout(DropoutRate)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1) # (D_Batch, D_Context, NEmbedding)
        out = self.proj(out) # (D_Batch, D_Context, NEmbedding)
        out = self.dropout(out) # (D_Batch, D_Context, NEmbedding)
        return out

In [11]:
class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(NEmbedding, 4 * NEmbedding),
            nn.ReLU(),
            nn.Linear(4 * NEmbedding, NEmbedding), # projection layer,
            nn.Dropout(DropoutRate),
        )

    def forward(self, x):
        return self.net(x)

In [12]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    def __init__(self):
        super().__init__()
        # Self-attention
        self.sa = MultiHeadAttention()
        # Feed-forward
        self.ffwd = FeedForward()
        # Layer-norm: https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html#torch.nn.LayerNorm
        self.ln1 = nn.LayerNorm(NEmbedding)
        self.ln2 = nn.LayerNorm(NEmbedding)

    def forward(self, x):
        # In "Attention is all you need", norm is applied after the tranformer
        # But Andrej Kaparthy mentioned that now it is more common to layernorm before
        # And we want residual network, so adding x
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))

        return x

In [13]:
class BigramLanguageModel(nn.Module):


    def __init__(self):
        
        super().__init__()

 
        # Token embedding
        self.token_embedding_table = nn.Embedding(NToken, NEmbedding)

        # Position embedding
        self.position_embedding_table = nn.Embedding(NContext, NEmbedding)

        # Final implemenation
        # What we want is nn.Sequential(Block_1, Block_2, Block_3, ... Block_NLayer)
        # We can first make a list [], then unpack
        self.blocks = nn.Sequential(
            *[Block() for _ in range(NLayer)]
        )

        # final layer norm
        self.ln_f = nn.LayerNorm(NEmbedding)

        # Calculate logits, returning back to Token
        self.lm_head = nn.Linear(NEmbedding, NToken)

    def forward(self, idx, targets=None):

        tok_emb = self.token_embedding_table(idx)

        # Embed [0, 1, ..., NContext-1]
        pos_emb = self.position_embedding_table(torch.arange(idx.shape[1], device=device))

        x = tok_emb + pos_emb
        
        x = self.blocks(x)

        x = self.ln_f(x)

        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            D_Batch, D_Context, D_Token = logits.shape
            logits = logits.view(D_Batch*D_Context, D_Token)
            targets = targets.view(D_Batch*D_Context)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # Let's make it work for batches
        # idx: (D_Batch, Current number of tokens generated)
        for _ in range(max_new_tokens):

            # Use the last NContext tokens
            idx_cond = idx[:, -NContext:]

            # (D_Batch, D_Context, NToken)
            logits, _ = self(idx_cond)
   
            # (D_Batch, "Last token", NToken)
            # -> (D_Batch, NToken)
            logits = logits[:, -1, :]

            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)

            # sample from the distribution using the prob for each row
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


# Now declare our model

In [14]:
m = BigramLanguageModel().to(device)
m = torch.compile(m)

In [15]:
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters()), 'parameters')

49386577 parameters


# Generate with the initial model

In [16]:
input_text = 'Dog'

tmp_start_text = torch.tensor(encode(input_text), device=device)
tmp_start_text = torch.unsqueeze(tmp_start_text, dim=0)

# generate from the model
print('Input text:')
print(input_text)

print()

tmp_generate = m.generate(tmp_start_text, max_new_tokens=10)
print('Generated text:')
print(decode(tmp_generate[0].tolist()))
# print(decode(m.generate(tmp_start_text, max_new_tokens=1000)[0].tolist()))

Input text:
Dog

Generated text:
Dog Screw functionpletedaminBLIC caféousandlpisible Einstein


# Prepare training

In [17]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=LearningRate)

In [18]:
def get_batch(split):

    """ Making Input and Target, depending ont split='train' or 'val' """

    # Select dataset
    data = data_train if split == 'train' else data_val

    # Randomly pick the starting index from the original text
    # We will make NBatch number of it
    ix = torch.randint(len(data) - NContext, (NBatch,))

    x = torch.stack([data[i:i+NContext] for i in ix])
    y = torch.stack([data[i+1:i+NContext+1] for i in ix])

    x = x.to(device)
    y = y.to(device)

    return x, y

In [19]:
# Function decorator
@torch.no_grad()
def estimate_loss():
    out = {}
    m.eval()
    for split in ['train', 'val']:
        # Make tensor placeholder
        losses = torch.zeros(EvalIteration)
        for k in range(EvalIteration):
            X, Y = get_batch(split)
            _, loss = m(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    m.train()
    return out

In [26]:
for iter in tqdm.tqdm(range(NMaxIteration)):

    if iter % EvalEvery == 0 or iter == NMaxIteration - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if device=='mps':
        torch.mps.synchronize()

  0%|          | 0/100 [00:00<?, ?it/s]

step 0: train loss 4.3331, val loss 5.0682


 99%|█████████▉| 99/100 [01:02<00:00,  4.30it/s] 

step 99: train loss 4.2660, val loss 4.9580


100%|██████████| 100/100 [01:36<00:00,  1.04it/s]


# Generate after the training

In [27]:
input_text = 'Dog'

tmp_start_text = torch.tensor(encode(input_text), device=device)
tmp_start_text = torch.unsqueeze(tmp_start_text, dim=0)

# generate from the model
print('Input text:')
print(input_text)

print()

tmp_generate = m.generate(tmp_start_text, max_new_tokens=1000)
print('Generated text:')
print(decode(tmp_generate[0].tolist()))

Input text:
Dog

Generated text:
Dog thy was
And king and war revoke, if very in mine eye wounds,
Before the awful its
My Duchess eyes and Oxford.

That now thou art queen, but not dance with
PAULINA:
Are now thy face I sin
QUERAKENdressed battle's point such friend
WICK:
Shall thy valour:
The firstOMPEYORK:
No, be sent me.
QUEEN MARGARINA:
The greaterance dry no?
You wearyUEENF despair to be set thy late of.

Where Lancaster; and art my heart
Be not beheld fence must I fear the Amazonu conflict with him so, thou shaltards powers for heaven.
Our sigh slaughter you shall break them daylight place, her arm:

Now fruit shall find't-head, and my true!
The l brotherCATESBY:
QUEEN ELIZABELLA sister of a shower.

In men may be laid pounds this boy shall not think him to us gracious services
Who is I have done was carried and Lewis tears, that 'tis honourbelike.
KING RICHARD II:

COMINCE:
Therefore are bound golden shallow.
Provethinks his string?
To am resisted of them take way?  but by one t